# Загрузка модели и датасета на Hugging Face

Ноутбук грузит большие файлы (~17 ГБ) через `upload_large_folder` — резюмируется после обрыва, использует быстрый Xet backend.

**Перед запуском:**
1. Вставьте свой токен в ячейку ниже (получить: https://huggingface.co/settings/tokens, права **Write**)
2. Укажите username и имена репозиториев
3. Положите файлы в папки `./model/` и `./data/` (или поменяйте пути)

## 1. Установка зависимостей

In [ ]:
%pip install -U huggingface_hub hf_xet

## 2. Настройки (правьте здесь)

In [ ]:
import os

# === ТОКЕН ===
HF_TOKEN = ""  # <-- ВСТАВЬТЕ СВОЙ ТОКЕН СЮДА

# === Куда заливаем ===
USERNAME = "kalDima"          # <-- ваш ник на HF (или название организации)
MODEL_REPO = f"{USERNAME}/pa_rl_smolvla"  # <-- имя репо модели
DATA_REPO  = f"{USERNAME}/libero_rollouts"   # <-- имя репо датасета

# === Что заливаем ===
MODEL_FOLDER = "./.model"   # папка с шардами модели
DATA_FOLDER  = "./.data"    # папка с датасетом

# === Параллельность аплоада ===
NUM_WORKERS = 8   # уменьшите до 4, если канал слабый

# Включаем быстрый режим Xet и пробрасываем токен в env (на всякий случай)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

## 3. Логин

In [ ]:
from huggingface_hub import login, whoami

login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in as:", whoami()["name"])

## 4. Проверка файлов
Показываем что нашлось — чтобы не залить пустую папку.

In [ ]:
from pathlib import Path

def show_folder(folder):
    p = Path(folder)
    if not p.exists():
        print(f"  [!] папка не найдена: {folder}")
        return
    total = 0
    for f in sorted(p.rglob("*")):
        if f.is_file():
            size_gb = f.stat().st_size / 1024**3
            total += f.stat().st_size
            print(f"  {f.relative_to(p)}  ({size_gb:.2f} GB)")
    print(f"  ИТОГО: {total / 1024**3:.2f} GB")

print(f"== {MODEL_FOLDER} ==")
show_folder(MODEL_FOLDER)
print(f"\n== {DATA_FOLDER} ==")
show_folder(DATA_FOLDER)

## 5. Создание репозиториев

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

api.create_repo(MODEL_REPO, repo_type="model",   exist_ok=True, private=False)
api.create_repo(DATA_REPO,  repo_type="dataset", exist_ok=True, private=False)

print(f"Model:   https://huggingface.co/{MODEL_REPO}")
print(f"Dataset: https://huggingface.co/datasets/{DATA_REPO}")

## 6. Заливка модели

Можно прервать (Kernel → Interrupt) и перезапустить ячейку — продолжит с того места, где остановилось. Уже залитые файлы пропустятся.

In [ ]:
api.upload_large_folder(
    repo_id=MODEL_REPO,
    repo_type="model",
    folder_path=MODEL_FOLDER,
    num_workers=NUM_WORKERS,
)
print("Готово:", f"https://huggingface.co/{MODEL_REPO}")

## 7. Заливка датасета

In [ ]:
api.upload_large_folder(
    repo_id=DATA_REPO,
    repo_type="dataset",
    folder_path=DATA_FOLDER,
    num_workers=NUM_WORKERS,
)
print("Готово:", f"https://huggingface.co/datasets/{DATA_REPO}")

## 8. (опционально) Заливка одиночного файла
Если нужно докинуть один файл — раскомментируйте и поправьте пути.

In [ ]:
api.upload_file(
    path_or_fileobj="./data/img_cache_critic_256.npz",
    path_in_repo="img_cache_critic_256.npz",
    repo_id=DATA_REPO,
    repo_type="dataset",
)

## ⚠️ Безопасность

В этом ноутбуке токен **захардкожен в коде**. Перед тем как куда-то его выкладывать:
- очистите ячейку с токеном (`Edit → Clear All Outputs` тоже не помешает),
- или замените на чтение из env: `HF_TOKEN = os.environ["HF_TOKEN"]`,
- если токен утёк — отзовите его на https://huggingface.co/settings/tokens и создайте новый.